# Data-Efficient Wheat Disease Detection Using Shifted Window Transformer

Official code for the paper by M. Khubaib, T. Kehkashan, M. Abdelhaq, M. A. Khan, M. Zaman, I. Ashraf, A. Rehman and A. Akhunzada, published in IEEE Transactions on Consumer Electronics, vol. 71, no. 3, pp. 9006–9020, 2025. DOI: [10.1109/TCE.2025.3582267](https://doi.org/10.1109/TCE.2025.3582267)

**Pipeline (follows Section III, Methodology, and Fig. 3)**
1. **Data acquisition (III-C, Table I):** three wheat datasets combined into 9 classes: Brown Rust, Crown and Root Rot, Fusarium Head Blight, Healthy, Leaf Rust, Loose Smut, Septoria, Stripe Rust, Yellow Rust.
2. **Duplicate removal (III-D-1):** SSIM similarity check (fast hash pre-filter, then SSIM confirmation).
3. **Preprocessing (III-D-2):** resize to 224×224, histogram equalization, normalization.
4. **Augmentation (III-D-3):** rotation, horizontal and vertical flips, brightness, Gaussian noise, elastic deformation.
5. **Model (III-B, III-E):** Swin Transformer `swin_tiny_patch4_window7_224` (4×4 patches, 96-dim embedding, shifted-window attention).
6. **Bayesian optimization (III-G, Table IV):** Gaussian-process Bayesian optimization with Expected Improvement, 10 iterations over learning rate, batch size and dropout.
7. **Training (III-F, III-H):** Adam, cosine annealing, 10 epochs, early stopping, 70/15/15 split, cross-entropy loss.
8. **Results (IV):** accuracy and loss curves (Figs. 4–5), confusion matrix (Fig. 6), Table V ablation, Table VI comparison, efficiency (Fig. 7), Grad-CAM (Fig. 8).

**Runs on:** Kaggle or Google Colab with a GPU (T4 or better).

### Datasets used in the paper
| Source | Paper ref. | Images | Where to get it |
|---|---|---|---|
| LWDCD 2020 (Large Wheat Disease Classification Dataset) | [64] | 4,321 | Kaggle search "LWDCD2020" |
| Wheat Leaf Disease Dataset | [65] | 2,615 | Kaggle `arindam235/wheat-leaf-disease-dataset` |
| Wheat Leaf Dataset | [66] | 2,410 | Mendeley Data `wgd66f8n6h/1` |

On Kaggle, attach the datasets to the notebook; every image folder under `/kaggle/input` is scanned. Elsewhere, put the folders in `DATA_DIRS`. Folder names are mapped to the 9 paper classes through `CLASS_ALIASES`, and any unmatched folder is reported so you can add an alias.

## 1. Setup

In [ ]:
!pip install -q timm kagglehub "optuna>=3.6" scikit-image opencv-python-headless

In [ ]:
import os, gc, re, json, math, time, random, warnings
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import optuna
from skimage.metrics import structural_similarity as ssim

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             classification_report)

warnings.filterwarnings("ignore")
print("torch", torch.__version__, "| timm", timm.__version__, "| optuna", optuna.__version__)

## 2. Configuration

In [ ]:
SEED = 42

# Data sources: list of folders. Empty -> scan /kaggle/input, or download KAGGLE_SOURCES with kagglehub.
DATA_DIRS = []
KAGGLE_SOURCES = ["arindam235/wheat-leaf-disease-dataset"]    # add LWDCD 2020 / other Kaggle slugs here

CLASS_NAMES = ["Brown Rust", "Crown and Root Rot", "Fusarium Head Blight", "Healthy", "Leaf Rust",
               "Loose Smut", "Septoria", "Stripe Rust", "Yellow Rust"]
CLASS_ALIASES = {   # normalized folder name -> paper class
    "brown rust": "Brown Rust", "brownrust": "Brown Rust",
    "crown and root rot": "Crown and Root Rot", "crown root rot": "Crown and Root Rot", "root rot": "Crown and Root Rot",
    "fusarium head blight": "Fusarium Head Blight", "fusarium": "Fusarium Head Blight", "head blight": "Fusarium Head Blight", "fhb": "Fusarium Head Blight",
    "healthy": "Healthy", "healthy wheat": "Healthy", "healthy leaf": "Healthy", "healthy leaves": "Healthy",
    "leaf rust": "Leaf Rust", "leafrust": "Leaf Rust",
    "loose smut": "Loose Smut", "wheat loose smut": "Loose Smut", "smut": "Loose Smut",
    "septoria": "Septoria", "septoria leaf blotch": "Septoria",
    "stripe rust": "Stripe Rust", "striperust": "Stripe Rust",
    "yellow rust": "Yellow Rust", "yellowrust": "Yellow Rust",
}

# Duplicate removal (III-D-1)
HASH_THRESH = 6        # Hamming distance (of 64 bits) to flag a candidate pair
SSIM_THRESH = 0.95     # SSIM at or above this = duplicate

# Split (III-F): 70 / 15 / 15
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
IMG_SIZE = 224

# Model and training (III-B, III-F, III-H)
MODEL_NAME = "swin_tiny_patch4_window7_224.ms_in1k"
EPOCHS = 10            # paper: 10 epochs
PATIENCE = 5           # paper: early stopping on validation loss
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

# Bayesian optimization (III-G, Table IV)
RUN_HPO = True
HPO_TRIALS = 10                        # paper: 10 iterations
HPO_EPOCHS = 3                         # short budget per trial
HPO_TRAIN_FRACTION = 0.5               # fraction of the training set used per trial
SEARCH_SPACE = {"lr": [1e-5, 1e-4, 1e-3], "batch_size": [16, 32, 64], "dropout": [0.1, 0.3, 0.5]}
PAPER_BEST = {"lr": 1e-4, "batch_size": 32, "dropout": 0.3}   # used when RUN_HPO = False

# Ablation study (Table V)
RUN_ABLATION = True

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_CLASSES = len(CLASS_NAMES)
print("Device:", DEVICE)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

## 3. Data acquisition (Section III-C, Table I)

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def map_class(folder):
    s = re.sub(r"\s+", " ", re.sub(r"[_\-\.]+", " ", folder.lower())).strip()
    for cand in (s, re.sub(r"^(\d+\s*)?(class\s*\d*\s*)?(wheat\s+)?", "", s), s.replace(" ", "")):
        if cand in CLASS_ALIASES:
            return CLASS_ALIASES[cand]
    return None

if not DATA_DIRS:
    if os.path.isdir("/kaggle/input"):
        DATA_DIRS = ["/kaggle/input"]
    else:
        import kagglehub
        DATA_DIRS = [kagglehub.dataset_download(s) for s in KAGGLE_SOURCES]

records, unmatched = [], {}
for root in DATA_DIRS:
    for dirpath, _, filenames in os.walk(root):
        imgs = [f for f in filenames if os.path.splitext(f)[1].lower() in IMG_EXT]
        if not imgs:
            continue
        folder = os.path.basename(dirpath)
        cls = map_class(folder)
        if cls is None:
            unmatched[dirpath] = len(imgs)
            continue
        records += [(os.path.join(dirpath, f), CLASS_NAMES.index(cls), cls) for f in imgs]

raw_df = pd.DataFrame(records, columns=["path", "label", "class"]).drop_duplicates("path").reset_index(drop=True)
if unmatched:
    print("Folders not mapped to a class (add to CLASS_ALIASES if needed):")
    for k, v in list(unmatched.items())[:30]:
        print(f"  {v:5d} images  {k}")
assert len(raw_df), "No images found - check DATA_DIRS / CLASS_ALIASES"

table1 = raw_df["class"].value_counts().reindex(CLASS_NAMES, fill_value=0).rename("# of Images (found)").to_frame()
table1["# of Images (paper)"] = [1256, 1021, 607, 2460, 1041, 930, 446, 208, 1395]
table1.loc["Total"] = table1.sum()
missing = [c for c in CLASS_NAMES if (raw_df["class"] == c).sum() == 0]
if missing:
    print("WARNING: no images for classes:", missing, "- attach the corresponding dataset(s).")
table1

In [ ]:
# Fig. 1: sample images per class
present = [c for c in CLASS_NAMES if (raw_df["class"] == c).any()]
fig, axes = plt.subplots(2, len(present), figsize=(2.4 * len(present), 5.2), squeeze=False)
for j, c in enumerate(present):
    for r, p in enumerate(raw_df[raw_df["class"] == c].sample(min(2, (raw_df["class"] == c).sum()), random_state=SEED).path):
        axes[r, j].imshow(Image.open(p).convert("RGB"))
    for r in range(2):
        axes[r, j].axis("off")
    axes[0, j].set_title(c, fontsize=9)
plt.suptitle("Sample wheat images per class"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig1_samples.png", dpi=300, bbox_inches="tight"); plt.show()

## 4. Duplicate removal with SSIM (Section III-D-1)
Comparing every pair with SSIM is too slow for about 9k images, so a 64-bit difference hash first finds candidate pairs. Each candidate pair is then confirmed with SSIM ≥ `SSIM_THRESH`, and only the first image of a duplicate pair is kept. Duplicates are removed **before** splitting, so no image leaks between train and test.

In [ ]:
def dhash(path, size=8):
    try:
        a = np.asarray(Image.open(path).convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    except Exception:
        return None
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten())

hashes = [dhash(p) for p in tqdm(raw_df.path, desc="hashing")]
ok = np.array([h is not None for h in hashes])
if (~ok).any():
    print(f"Unreadable images dropped: {(~ok).sum()}")
raw_df = raw_df[ok].reset_index(drop=True)
H = np.stack([h for h in hashes if h is not None])

POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
cand_pairs = []
for s in tqdm(range(0, len(H), 512), desc="candidate pairs"):
    d = POPCOUNT[H[s:s + 512, None, :] ^ H[None, :, :]].sum(-1, dtype=np.int16)
    i, j = np.nonzero(d <= HASH_THRESH)
    i = i + s
    keep = i < j
    cand_pairs += list(zip(i[keep], j[keep]))
print("Candidate pairs:", len(cand_pairs))

@lru_cache(maxsize=4096)
def small_gray(idx):
    return np.asarray(Image.open(raw_df.path[idx]).convert("L").resize((128, 128), Image.BILINEAR))

removed, dup_log = set(), []
for i, j in tqdm(cand_pairs, desc="SSIM check"):
    if i in removed or j in removed:
        continue
    score = ssim(small_gray(i), small_gray(j), data_range=255)
    if score >= SSIM_THRESH:
        removed.add(j)
        dup_log.append((raw_df.path[i], raw_df.path[j], raw_df["class"][i], raw_df["class"][j], score))

df = raw_df.drop(index=sorted(removed)).reset_index(drop=True)
dups = pd.DataFrame(dup_log, columns=["kept", "removed", "class_kept", "class_removed", "ssim"])
dups.to_csv(f"{OUT_DIR}/duplicates_removed.csv", index=False)
print(f"Removed {len(removed)} duplicates ({(dups.class_kept != dups.class_removed).sum()} across different classes)")
print(f"Images after de-duplication: {len(df)}")

### Train / validation / test split (70 / 15 / 15, stratified)

In [ ]:
train_df, temp_df = train_test_split(df, train_size=TRAIN_FRAC, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=temp_df.label, random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]

split_table = pd.DataFrame({n: d["class"].value_counts().reindex(CLASS_NAMES, fill_value=0)
                            for n, d in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]})
split_table.loc["Total"] = split_table.sum()
split_table

## 5. Preprocessing and augmentation (Section III-D-2, III-D-3)

In [ ]:
def hist_equalize(img):
    # histogram equalization on the luminance channel (keeps disease colour)
    a = np.asarray(img.convert("RGB"))
    ycc = cv2.cvtColor(a, cv2.COLOR_RGB2YCrCb)
    ycc[..., 0] = cv2.equalizeHist(ycc[..., 0])
    return Image.fromarray(cv2.cvtColor(ycc, cv2.COLOR_YCrCb2RGB))

class AddGaussianNoise:
    def __init__(self, std=0.03):
        self.std = std
    def __call__(self, t):
        return (t + self.std * torch.randn_like(t)).clamp(0, 1)

def get_transforms(model):
    cfg = timm.data.resolve_model_data_config(model)
    mean, std = cfg["mean"], cfg["std"]
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(20),                                          # rotation
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),     # flipping
        transforms.ColorJitter(brightness=0.25),                                # brightness adjustment
        transforms.RandomApply([transforms.ElasticTransform(alpha=30.0, sigma=5.0)], p=0.25),  # elastic deformation
        transforms.ToTensor(),
        transforms.RandomApply([AddGaussianNoise(0.03)], p=0.3),                # Gaussian noise
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_tf, eval_tf

class WheatDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths, self.labels, self.transform = frame.path.tolist(), frame.label.tolist(), transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = hist_equalize(Image.open(self.paths[i]).convert("RGB"))
        return self.transform(img), self.labels[i]

def make_loader(frame, transform, batch_size, shuffle=False):
    return DataLoader(WheatDataset(frame, transform), batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=USE_AMP, drop_last=shuffle and len(frame) > batch_size,
                      generator=torch.Generator().manual_seed(SEED) if shuffle else None)

In [ ]:
# Preprocessing preview: original -> histogram equalized -> augmented
_m = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
_train_tf, _ = get_transforms(_m)
cfg = timm.data.resolve_model_data_config(_m); del _m
inv = transforms.Normalize([-m / s for m, s in zip(cfg["mean"], cfg["std"])], [1 / s for s in cfg["std"]])
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for j, p in enumerate(train_df.sample(4, random_state=SEED).path):
    im = Image.open(p).convert("RGB"); eq = hist_equalize(im)
    axes[0, j].imshow(im.resize((IMG_SIZE, IMG_SIZE))); axes[1, j].imshow(eq.resize((IMG_SIZE, IMG_SIZE)))
    axes[2, j].imshow(inv(_train_tf(eq)).permute(1, 2, 0).clamp(0, 1).numpy())
for r, t in enumerate(["Original", "Histogram equalized", "Augmented"]):
    axes[r, 0].set_ylabel(t, fontsize=11)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/preprocessing_examples.png", dpi=300, bbox_inches="tight"); plt.show()

## 6. Model, training and evaluation utilities (Sections III-B, III-E, III-F)
Swin-Tiny with a 9-class softmax head, cross-entropy loss, Adam with cosine annealing, and early stopping on validation loss.

In [ ]:
criterion = nn.CrossEntropyLoss()

def build_model(name=MODEL_NAME, dropout=0.3, pretrained=True):
    return timm.create_model(name, pretrained=pretrained, num_classes=NUM_CLASSES, drop_rate=dropout).to(DEVICE)

def run_epoch(model, loader, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in tqdm(loader, leave=False, desc="train" if train else "eval"):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

def train_model(model, train_frame, lr, batch_size, epochs=EPOCHS, patience=PATIENCE, ckpt=None, verbose=True):
    train_tf, eval_tf = get_transforms(model)
    train_loader = make_loader(train_frame, train_tf, batch_size, shuffle=True)
    val_loader = make_loader(val_df, eval_tf, batch_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)   # cosine annealing
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_loss, best_state, wait = float("inf"), None, 0
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, scaler)
        va_loss, va_acc = run_epoch(model, val_loader)
        scheduler.step()
        for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc]):
            history[k].append(v)
        improved = va_loss < best_loss
        if improved:
            best_loss, wait = va_loss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        if verbose:
            print(f"Epoch {epoch:02d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f}" + ("  *best*" if improved else ""))
        if wait >= patience:
            if verbose:
                print(f"Early stopping (no val-loss improvement for {patience} epochs).")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    if ckpt and best_state is not None:
        torch.save(best_state, ckpt)
    return model, history

@torch.no_grad()
def predict(model, frame, batch_size=64):
    _, eval_tf = get_transforms(model)
    model.eval()
    ys, ps = [], []
    for x, y in make_loader(frame, eval_tf, batch_size):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x.to(DEVICE))
        ps.append(logits.float().softmax(1).cpu()); ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

def scores(y, prob):
    pred = prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    return {"Accuracy": accuracy_score(y, pred), "Precision": p, "Recall": r, "F1": f}

## 7. Bayesian optimization of hyperparameters (Section III-G, Table IV)
Gaussian-process Bayesian optimization (`optuna.samplers.GPSampler`, Expected-Improvement acquisition) runs for 10 iterations over the Table IV search space and maximizes validation accuracy. Each trial trains briefly on part of the training set to keep the cost reasonable.

In [ ]:
hpo_train = train_df.groupby("label", group_keys=False).apply(
    lambda d: d.sample(frac=HPO_TRAIN_FRACTION, random_state=SEED)).reset_index(drop=True)

def objective(trial):
    lr = trial.suggest_categorical("lr", SEARCH_SPACE["lr"])
    bs = trial.suggest_categorical("batch_size", SEARCH_SPACE["batch_size"])
    dr = trial.suggest_categorical("dropout", SEARCH_SPACE["dropout"])
    seed_everything(SEED)
    m = build_model(dropout=dr)
    m, h = train_model(m, hpo_train, lr=lr, batch_size=bs, epochs=HPO_EPOCHS, patience=HPO_EPOCHS, verbose=False)
    acc = max(h["val_acc"])
    print(f"Trial {trial.number}: lr={lr:g}, batch={bs}, dropout={dr} -> val acc {acc:.4f}")
    del m
    gc.collect(); torch.cuda.empty_cache()
    return acc

if RUN_HPO:
    sampler = optuna.samplers.GPSampler(seed=SEED, n_startup_trials=3)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=HPO_TRIALS)
    BEST = study.best_params
    hpo_log = study.trials_dataframe()[["number", "params_lr", "params_batch_size", "params_dropout", "value"]]
    hpo_log.to_csv(f"{OUT_DIR}/bayesian_optimization_trials.csv", index=False)
else:
    BEST = dict(PAPER_BEST)

table4 = pd.DataFrame({"Parameter": ["Learning Rate", "Batch Size", "Dropout Rate"],
                       "Search Range": [str(SEARCH_SPACE["lr"]), str(SEARCH_SPACE["batch_size"]), str(SEARCH_SPACE["dropout"])],
                       "Optimal Value (this run)": [BEST["lr"], BEST["batch_size"], BEST["dropout"]],
                       "Optimal Value (paper)": [PAPER_BEST["lr"], PAPER_BEST["batch_size"], PAPER_BEST["dropout"]]})
table4.to_csv(f"{OUT_DIR}/table4_hyperparameters.csv", index=False)
table4

In [ ]:
if RUN_HPO:
    vals = study.trials_dataframe()["value"].values
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(vals) + 1), vals, "o-", label="Trial val accuracy")
    plt.plot(range(1, len(vals) + 1), np.maximum.accumulate(vals), "s--", label="Best so far")
    plt.xlabel("Iteration"); plt.ylabel("Validation accuracy"); plt.title("Bayesian optimization (GP + EI)")
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/bayesian_optimization.png", dpi=300, bbox_inches="tight"); plt.show()

## 8. Final training of the Swin Transformer (Section III-H)

In [ ]:
seed_everything(SEED)
model = build_model(dropout=BEST["dropout"])
print(f"Swin-Tiny parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M | "
      f"patch size 4, embed dim {model.patch_embed.proj.out_channels}, window 7")
BEST_PATH = f"{OUT_DIR}/swin_tiny_wheat_best.pth"
model, history = train_model(model, train_df, lr=BEST["lr"], batch_size=BEST["batch_size"], ckpt=BEST_PATH)
print("Best model saved to", BEST_PATH)

In [ ]:
# Fig. 4: training and validation accuracy | Fig. 5: training and validation loss
ep = range(1, len(history["train_loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(ep, np.array(history["train_acc"]) * 100, "o-", label="Training Accuracy")
ax[0].plot(ep, np.array(history["val_acc"]) * 100, "s-", label="Validation Accuracy")
ax[0].set_title("Training and Validation Accuracy"); ax[0].set_xlabel("Epochs"); ax[0].set_ylabel("Accuracy (%)")
ax[1].plot(ep, history["train_loss"], "o-", color="red", label="Training Loss")
ax[1].plot(ep, history["val_loss"], "s-", color="green", label="Validation Loss")
ax[1].set_title("Training and Validation Loss"); ax[1].set_xlabel("Epochs"); ax[1].set_ylabel("Loss")
for a in ax:
    a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4_fig5_curves.png", dpi=300, bbox_inches="tight"); plt.show()

## 9. Results (Section IV)
Test-set accuracy plus macro-averaged precision, recall and F1, with a per-class report.

In [ ]:
y_true, y_prob = predict(model, test_df)
y_pred = y_prob.argmax(1)
swin_scores = scores(y_true, y_prob)
for k, v in swin_scores.items():
    print(f"{k:9s}: {v*100:.2f}%")
labels_present = sorted(np.unique(y_true))
print("\n" + classification_report(y_true, y_pred, labels=labels_present,
                                   target_names=[CLASS_NAMES[i] for i in labels_present], digits=4, zero_division=0))
pd.DataFrame(classification_report(y_true, y_pred, labels=labels_present, target_names=[CLASS_NAMES[i] for i in labels_present],
                                   output_dict=True, zero_division=0)).T.to_csv(f"{OUT_DIR}/per_class_report.csv")

In [ ]:
# Fig. 6: confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=labels_present)
names = [CLASS_NAMES[i] for i in labels_present]
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix: Swin Transformer")
plt.xticks(rotation=40, ha="right"); plt.yticks(rotation=0); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig6_confusion_matrix.png", dpi=300, bbox_inches="tight"); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
if off.sum():
    i, j = np.unravel_index(off.argmax(), off.shape)
    print(f"Most frequent confusion: {names[i]} -> {names[j]} ({off[i, j]} images)")

## 10. Ablation study (Section IV-A, Table V)
Each variant is trained with the same recipe and hyperparameters.
- **No Shifted Window Attention:** Swin-Tiny with every window shift set to 0, so attention stays inside fixed windows.
- **No Hierarchical Structure:** a flat, single-scale transformer of similar size (ViT-Small/16, no patch merging), used as a stand-in.
- **No Patch-based Tokenization:** a convolutional network working on pixels (ResNet-50), used as a stand-in.

The paper does not say exactly how the last two components were removed, so those rows use the stand-in models above.

In [ ]:
def disable_shifted_windows(m):
    n = 0
    for blk in m.modules():
        if blk.__class__.__name__ == "SwinTransformerBlock":
            blk.shift_size = (0, 0) if isinstance(blk.shift_size, (tuple, list)) else 0
            if getattr(blk, "attn_mask", None) is not None:
                blk.attn_mask = None
            if hasattr(blk, "dynamic_mask"):
                blk.dynamic_mask = False
            n += 1
    return n

ABLATIONS = [
    ("No Shifted Window Attention", MODEL_NAME, True),
    ("No Hierarchical Structure (ViT-S/16)", "vit_small_patch16_224.augreg_in21k_ft_in1k", False),
    ("No Patch-based Tokenization (ResNet-50)", "resnet50.a1_in1k", False),
]
ablation_models = {"Full Model (Swin Transformer)": model}
table5_rows = [{"Model Variant": "Full Model (Swin Transformer)", **{k: round(v * 100, 2) for k, v in swin_scores.items()}}]

if RUN_ABLATION:
    for name, arch, no_shift in ABLATIONS:
        print(f"\n=== {name} ===")
        seed_everything(SEED)
        m = build_model(arch, dropout=BEST["dropout"])
        if no_shift:
            print("Blocks with shift disabled:", disable_shifted_windows(m))
        m, _ = train_model(m, train_df, lr=BEST["lr"], batch_size=BEST["batch_size"], verbose=True)
        yt, pr = predict(m, test_df)
        table5_rows.append({"Model Variant": name, **{k: round(v * 100, 2) for k, v in scores(yt, pr).items()}})
        ablation_models[name] = m.cpu()
        gc.collect(); torch.cuda.empty_cache()

table5 = pd.DataFrame(table5_rows)
table5["Paper Accuracy (%)"] = [99.30, 95.40, 96.10, 96.80][:len(table5)]
table5.to_csv(f"{OUT_DIR}/table5_ablation.csv", index=False)
table5

## 11. Comparison with state-of-the-art (Section IV-B, Table VI)
Previous-study values are taken from Table VI of the paper; the first row comes from this run.

In [ ]:
table6 = pd.DataFrame([
    ["Proposed Model (Swin Transformer), this run", *[round(swin_scores[k] * 100, 2) for k in ["Accuracy", "Precision", "Recall", "F1"]]],
    ["DCNN (Baseline) [65]", 97.88, 97.60, 97.80, 97.50],
    ["E-MMC [69]",           94.16, 93.25, 93.66, 94.20],
    ["Hybrid [70]",          98.76, 98.77, 98.81, 98.79],
    ["DACNN [71]",           95.18, 94.99, 95.10, 95.21],
    ["DT and CNN [72]",      97.20, 96.33, 96.68, 97.20],
    ["CNN + ViT [23]",       99.83, 98.33, 98.68, 99.45],
    ["CNN + ViT [21]",      100.00, 99.88, 99.45, 99.69],
], columns=["Model", "Accuracy (%)", "Precision (%)", "Recall (%)", "F1 Score (%)"])
table6.to_csv(f"{OUT_DIR}/table6_sota.csv", index=False)
table6

## 12. Efficiency: accuracy, FLOPs and inference time (Fig. 7)

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

def efficiency(m, runs=50):
    m = m.to(DEVICE).eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        m(x)
    gflops = fc.get_total_flops() / 1e9
    with torch.no_grad():
        for _ in range(10):
            m(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs):
            m(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / runs * 1000
    params = sum(p.numel() for p in m.parameters()) / 1e6
    m.cpu()
    return params, gflops, ms

eff_rows = []
for (name, m), acc in zip(ablation_models.items(), table5["Accuracy"]):
    params, gflops, ms = efficiency(m)
    eff_rows.append({"Model": name, "Accuracy (%)": acc, "Params (M)": round(params, 2),
                     "GFLOPs": round(gflops, 2), "Inference (ms/img)": round(ms, 2)})
eff = pd.DataFrame(eff_rows)
eff.to_csv(f"{OUT_DIR}/fig7_efficiency.csv", index=False)
model.to(DEVICE)

norm = eff.set_index("Model")[["Accuracy (%)", "GFLOPs", "Inference (ms/img)"]]
norm = norm / norm.max()
ax = norm.plot(kind="bar", figsize=(11, 5), width=0.8)
ax.set_ylabel("Normalized value (max = 1)"); ax.set_title("Performance and efficiency comparison")
plt.xticks(rotation=15, ha="right"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig7_efficiency.png", dpi=300, bbox_inches="tight"); plt.show()
eff

## 13. Grad-CAM interpretability (Fig. 8)
Grad-CAM on the last Swin stage (a 7×7 token grid) highlights the leaf regions that drive each prediction.

In [ ]:
def grad_cam(m, img_tensor, class_index=None, target_layer=None):
    m.eval()
    target_layer = target_layer or m.layers[-1]
    store = {}
    def hook(module, inputs, output):
        store["act"] = output
        output.register_hook(lambda g: store.__setitem__("grad", g))
    h = target_layer.register_forward_hook(hook)
    try:
        m.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = m(img_tensor.unsqueeze(0).to(DEVICE))
            probs = logits.softmax(1)[0].detach().cpu().numpy()
            class_index = int(probs.argmax()) if class_index is None else class_index
            logits[0, class_index].backward()
    finally:
        h.remove()
    act, grad = store["act"][0].detach(), store["grad"][0].detach()
    if act.dim() == 3 and act.shape[0] < act.shape[-1]:          # Swin stages are channels-last (H, W, C)
        act, grad = act.permute(2, 0, 1), grad.permute(2, 0, 1)
    cam = F.relu((grad.mean(dim=(1, 2), keepdim=True) * act).sum(0)).cpu().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), class_index, probs

_, eval_tf = get_transforms(model)
test_eval = test_df.assign(pred=y_pred, confidence=y_prob.max(1))

def show_gradcam(frame, n=6, fname="fig8_gradcam.png", title="Grad-CAM visualizations (Swin Transformer)"):
    picks = frame.groupby("label", group_keys=False).apply(lambda d: d.sample(1, random_state=SEED)).head(n)
    fig, axes = plt.subplots(2, len(picks), figsize=(3.2 * len(picks), 6.6), squeeze=False)
    for j, row in enumerate(picks.itertuples()):
        img = hist_equalize(Image.open(row.path).convert("RGB")).resize((IMG_SIZE, IMG_SIZE))
        cam, pred, probs = grad_cam(model, eval_tf(img))
        cam_up = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        axes[0, j].imshow(img); axes[0, j].set_title(f"True: {CLASS_NAMES[row.label]}", fontsize=9)
        axes[1, j].imshow(img); axes[1, j].imshow(cam_up, cmap="jet", alpha=0.45)
        axes[1, j].set_title(f"Pred: {CLASS_NAMES[pred]} ({probs[pred]:.2f})", fontsize=9,
                             color="green" if pred == row.label else "red")
        for a in axes[:, j]:
            a.axis("off")
    plt.suptitle(title); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{fname}", dpi=300, bbox_inches="tight"); plt.show()

show_gradcam(test_eval, n=NUM_CLASSES)
wrong = test_eval[test_eval.label != test_eval.pred]
print("Misclassified test images:", len(wrong))
if len(wrong):
    show_gradcam(wrong, n=6, fname="gradcam_misclassified.png", title="Grad-CAM of misclassified images")

## 14. Save results

In [ ]:
test_eval.to_csv(f"{OUT_DIR}/test_predictions.csv", index=False)
results = {
    "model": MODEL_NAME, "best_hyperparameters": BEST, "epochs_trained": len(history["train_loss"]),
    "images": {"found": int(len(raw_df)), "duplicates_removed": int(len(removed)), "used": int(len(df))},
    "split": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "test_metrics_percent": {k: round(v * 100, 2) for k, v in swin_scores.items()},
    "confusion_matrix": cm.tolist(), "history": history,
    "ablation": table5.to_dict(orient="records"), "efficiency": eff.to_dict(orient="records"),
}
with open(f"{OUT_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2, default=float)
print(json.dumps(results["test_metrics_percent"], indent=2))
print("Saved to:", sorted(os.listdir(OUT_DIR)))